# Data Engineering Pipeline V2 - Fault-Tolerant Execution

This notebook replaces the duplicated and incomplete V2 implementation with one configuration-driven ETL pipeline. It processes every source dataset directly under `data/`, writes clean outputs to `data/processed_v3/`, creates enriched business fact tables and monthly summaries, and records data-quality and run metrics.

### V1 to V2 improvements
- One reusable extractor, transformer, validator, loader, and runner instead of repeated dataset-specific classes.
- Correct method names, exception behavior, type coercion, date parsing, and JSON extraction.
- Dataset-grain-aware duplicate removal (raw orders use `order_id + product_id`, not only `order_id`).
- Explicit quality rules and referential-integrity checks.
- Business-ready revenue, profit, customer, payment, shipping, and location enrichment.
- No optional SQLAlchemy or Parquet dependency is required.


In [1]:
from __future__ import annotations

import json
import logging
import re
import time
import traceback
import uuid
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("pipeline_v2")

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_DIR / "processed_v3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
print(f"Reading sources from: {DATA_DIR.resolve()}")
print(f"Writing outputs to: {OUTPUT_DIR.resolve()}")


Reading sources from: C:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\data
Writing outputs to: C:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\data\processed_v3


## Reusable ETL components

The components below keep I/O, transformation, validation, and orchestration separate. Dataset-specific behavior lives in configuration rather than duplicated pipeline blocks.


In [2]:
class BaseExtractor(ABC):
    @abstractmethod
    def extract(self) -> pd.DataFrame:
        raise NotImplementedError


class BaseTransformer(ABC):
    @abstractmethod
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        raise NotImplementedError


class BaseValidator(ABC):
    @abstractmethod
    def validate(self, df: pd.DataFrame) -> list[dict[str, Any]]:
        raise NotImplementedError


class BaseLoader(ABC):
    @abstractmethod
    def load(self, df: pd.DataFrame) -> Path:
        raise NotImplementedError


class FileExtractor(BaseExtractor):
    """Read CSV or record-oriented JSON based on the file extension."""

    def __init__(self, path: Path):
        self.path = Path(path)

    def extract(self) -> pd.DataFrame:
        if not self.path.exists():
            raise FileNotFoundError(f"Source file not found: {self.path}")
        if self.path.suffix.lower() == ".csv":
            df = pd.read_csv(self.path, low_memory=False)
        elif self.path.suffix.lower() == ".json":
            df = pd.read_json(self.path, orient="records")
        else:
            raise ValueError(f"Unsupported source format: {self.path.suffix}")
        log.info("Extracted %s: %d rows, %d columns", self.path.name, *df.shape)
        return df


class CSVLoader(BaseLoader):
    """Write deterministic, portable CSV output."""

    def __init__(self, path: Path):
        self.path = Path(path)

    def load(self, df: pd.DataFrame) -> Path:
        self.path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(self.path, index=False, encoding="utf-8", date_format="%Y-%m-%d")
        log.info("Loaded %d rows to %s", len(df), self.path)
        return self.path


In [3]:
@dataclass(frozen=True)
class DatasetSpec:
    name: str
    filename: str
    key: tuple[str, ...]
    required: tuple[str, ...]
    string_columns: tuple[str, ...] = ()
    lowercase_columns: tuple[str, ...] = ()
    numeric_columns: dict[str, str] = field(default_factory=dict)
    date_columns: tuple[str, ...] = ()
    non_negative: tuple[str, ...] = ()
    ranges: dict[str, tuple[float, float]] = field(default_factory=dict)
    allowed_values: dict[str, tuple[str, ...]] = field(default_factory=dict)


def snake_case(value: Any) -> str:
    value = re.sub(r"[^0-9a-zA-Z]+", "_", str(value).strip())
    return value.strip("_").lower()


class DatasetTransformer(BaseTransformer):
    def __init__(self, spec: DatasetSpec):
        self.spec = spec
        self.rejected_rows = pd.DataFrame()
        self.quality_checks: list[dict[str, Any]] = []

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        result.columns = [snake_case(column) for column in result.columns]

        missing = sorted(set(self.spec.required) - set(result.columns))
        if missing:
            raise ValueError(f"{self.spec.name}: missing required columns {missing}")

        for column in self.spec.string_columns:
            if column in result:
                result[column] = result[column].astype("string").str.strip()
        for column in self.spec.lowercase_columns:
            if column in result:
                result[column] = result[column].str.lower()
        for column, dtype in self.spec.numeric_columns.items():
            if column in result:
                values = pd.to_numeric(result[column], errors="coerce")
                result[column] = values.astype(dtype)
        for column in self.spec.date_columns:
            if column in result:
                result[column] = pd.to_datetime(result[column], errors="coerce")

        # Quarantine invalid measures instead of aborting the full pipeline.
        reject_mask = pd.Series(False, index=result.index)
        rejection_reason = pd.Series("", index=result.index, dtype="string")
        for column in self.spec.non_negative:
            if column in result:
                invalid = result[column].notna() & result[column].lt(0)
                count = int(invalid.sum())
                if count:
                    rejection_reason.loc[invalid] += f"{column} is negative; "
                    reject_mask |= invalid
                self.quality_checks.append({"dataset": self.spec.name, "check": f"quarantined_non_negative:{column}", "severity": "WARNING", "failed_rows": count, "status": "PASS" if count == 0 else "FAIL"})
        for column, (minimum, maximum) in self.spec.ranges.items():
            if column in result:
                invalid = result[column].notna() & ~result[column].between(minimum, maximum)
                count = int(invalid.sum())
                if count:
                    rejection_reason.loc[invalid] += f"{column} is outside [{minimum}, {maximum}]; "
                    reject_mask |= invalid
                self.quality_checks.append({"dataset": self.spec.name, "check": f"quarantined_range:{column}", "severity": "WARNING", "failed_rows": count, "status": "PASS" if count == 0 else "FAIL"})
        self.rejected_rows = result.loc[reject_mask].copy()
        if not self.rejected_rows.empty:
            self.rejected_rows["rejection_reason"] = rejection_reason.loc[reject_mask].str.rstrip("; " )
            log.warning("%s: quarantined %d invalid rows", self.spec.name, len(self.rejected_rows))
        result = result.loc[~reject_mask].copy()

        before = len(result)
        result = result.drop_duplicates(subset=list(self.spec.key), keep="first").reset_index(drop=True)
        log.info("%s: removed %d duplicate key rows", self.spec.name, before - len(result))
        return result


class DatasetValidator(BaseValidator):
    """Return a quality report and fail only on critical schema/key defects."""

    def __init__(self, spec: DatasetSpec):
        self.spec = spec

    def validate(self, df: pd.DataFrame) -> list[dict[str, Any]]:
        checks: list[dict[str, Any]] = []

        def record(check: str, failures: int, severity: str = "ERROR") -> None:
            checks.append({
                "dataset": self.spec.name,
                "check": check,
                "severity": severity,
                "failed_rows": int(failures),
                "status": "PASS" if failures == 0 else "FAIL",
            })

        record("dataset_not_empty", int(df.empty))
        for column in self.spec.required:
            # Only missing record identifiers are critical; attributes such as email are warnings.
            severity = "ERROR" if column in self.spec.key else "WARNING"
            record(f"not_null:{column}", df[column].isna().sum(), severity=severity)
        record("unique_key", df.duplicated(list(self.spec.key)).sum())
        for column in self.spec.non_negative:
            if column in df:
                record(f"non_negative:{column}", (df[column].dropna() < 0).sum())
        for column, (minimum, maximum) in self.spec.ranges.items():
            if column in df:
                invalid = df[column].dropna().lt(minimum) | df[column].dropna().gt(maximum)
                record(f"range:{column}[{minimum},{maximum}]", invalid.sum())
        for column, allowed in self.spec.allowed_values.items():
            if column in df:
                invalid = ~df[column].dropna().isin(allowed)
                record(f"allowed_values:{column}", invalid.sum(), severity="WARNING")

        critical = [c for c in checks if c["severity"] == "ERROR" and c["status"] == "FAIL"]
        if critical:
            log.error("%s: captured %d critical quality failures; execution will continue", self.spec.name, len(critical))
        return checks


## Dataset contracts

Only files directly under `data/` are sources. Existing `data/raw/` and `data/clean/` directories are treated as historical mirrors/outputs so records are not processed twice.


In [4]:
SPECS = [
    DatasetSpec("raw_orders", "raw_orders.csv", ("order_id", "product_id"),
                ("order_id", "customer_id", "product_id", "order_date", "quantity"),
                ("order_id", "customer_id", "product_id", "payment_type", "payment_provider", "shipping_carrier", "shipping_speed"),
                ("payment_type", "payment_provider", "shipping_carrier", "shipping_speed"),
                {"quantity": "Int64", "discount": "Float64", "shipping_cost": "Float64"}, ("order_date",),
                ("quantity", "discount", "shipping_cost"), {"discount": (0, 100)}),
    DatasetSpec("raw_customers", "raw_customers.json", ("customer_id",), ("customer_id", "signup_date"),
                ("customer_id", "first_name", "last_name", "email", "country"), ("email", "country"),
                date_columns=("signup_date",)),
    DatasetSpec("raw_products", "raw_products.csv", ("product_id",), ("product_id", "product_name", "price"),
                ("product_id", "product_name", "category"), ("category",),
                {"price": "Float64", "stock_quantity": "Int64"}, non_negative=("price", "stock_quantity")),
    DatasetSpec("orders", "orders.csv", ("order_id",), ("order_id", "customer_id", "order_date", "payment_id"),
                ("order_status",), ("order_status",), date_columns=("order_date", "shipping_date"),
                allowed_values={"order_status": ("processing", "shipped", "delivered", "cancelled", "returned")}),
    DatasetSpec("customers", "customers.csv", ("customer_id",), ("customer_id", "email", "registration_date"),
                ("first_name", "last_name", "email", "phone", "gender"), ("email", "gender"),
                date_columns=("date_of_birth", "registration_date"),
                allowed_values={"gender": ("m", "f", "u")}),
    DatasetSpec("products", "products.csv", ("product_id",), ("product_id", "product_name", "unit_price"),
                ("product_name", "category", "brand"), ("category",),
                {"unit_price": "Float64", "cost_price": "Float64"}, non_negative=("unit_price", "cost_price")),
    DatasetSpec("payments", "payments.json", ("payment_id",), ("payment_id", "payment_method", "payment_status", "payment_date"),
                ("payment_method", "payment_status"), ("payment_method", "payment_status"), date_columns=("payment_date",),
                allowed_values={"payment_status": ("pending", "completed", "failed", "refunded"),
                                "payment_method": ("credit card", "debit card", "paypal", "bank transfer")}),
    DatasetSpec("locations", "locations.csv", ("location_id",), ("location_id", "country"),
                ("city", "state", "country", "postal_code"), ("country",)),
    DatasetSpec("shipping", "shipping.csv", ("shipment_id",), ("shipment_id", "order_id", "location_id", "carrier"),
                ("carrier",), ("carrier",), {"shipping_cost": "Float64"}, ("delivery_date",), ("shipping_cost",),
                allowed_values={"carrier": ("fedex", "ups", "dhl", "usps", "other")}),
    DatasetSpec("order_items", "order_items.csv", ("order_item_id",), ("order_item_id", "order_id", "product_id", "quantity", "selling_price"),
                numeric_columns={"quantity": "Int64", "selling_price": "Float64"},
                non_negative=("quantity", "selling_price")),
]

assert len({spec.filename for spec in SPECS}) == len(SPECS), "Each source must be configured once"
assert {p.name for p in DATA_DIR.iterdir() if p.is_file() and p.suffix.lower() in {".csv", ".json"}} == {spec.filename for spec in SPECS}, "Update SPECS when source files change"
pd.DataFrame([{"dataset": s.name, "source": s.filename, "key": ", ".join(s.key)} for s in SPECS])


,dataset,source,key
0,raw_orders,raw_orders.csv,"order_id, product_id"
1,raw_customers,raw_customers.json,customer_id
2,raw_products,raw_products.csv,product_id
3,orders,orders.csv,order_id
4,customers,customers.csv,customer_id
5,products,products.csv,product_id
6,payments,payments.json,payment_id
7,locations,locations.csv,location_id
8,shipping,shipping.csv,shipment_id
9,order_items,order_items.csv,order_item_id


In [5]:
class ETLPipelineRunner:
    def __init__(self, spec: DatasetSpec):
        self.spec = spec
        self.extractor = FileExtractor(DATA_DIR / spec.filename)
        self.transformer = DatasetTransformer(spec)
        self.validator = DatasetValidator(spec)
        self.loader = CSVLoader(OUTPUT_DIR / f"{spec.name}.csv")

    def run(self) -> tuple[pd.DataFrame | None, dict[str, Any], list[dict[str, Any]]]:
        started = time.perf_counter()
        metrics = {
            "run_id": str(uuid.uuid4()), "dataset": self.spec.name, "status": "FAILED",
            "source_rows": 0, "output_rows": 0, "duplicates_removed": 0,
            "output": None, "elapsed_seconds": 0.0, "error": None,
        }
        quality: list[dict[str, Any]] = []
        try:
            source = self.extractor.extract()
            metrics["source_rows"] = len(source)
            clean = self.transformer.transform(source)
            quality = self.validator.validate(clean)
            output = self.loader.load(clean)
            metrics.update({"status": "SUCCESS", "output_rows": len(clean),
                            "duplicates_removed": len(source) - len(clean), "output": str(output)})
            return clean, metrics, quality
        except Exception as exc:
            error_message = f"{type(exc).__name__}: {exc}"
            metrics["error"] = error_message
            quality.append({"dataset": self.spec.name, "check": "operational_execution",
                            "severity": "ERROR", "failed_rows": 0, "status": "FAIL",
                            "details": error_message})
            log.error("Captured pipeline failure for %s; continuing with remaining datasets\n%s", self.spec.name, traceback.format_exc())
            return None, metrics, quality
        finally:
            metrics["elapsed_seconds"] = round(time.perf_counter() - started, 4)


datasets: dict[str, pd.DataFrame] = {}
run_metrics: list[dict[str, Any]] = []
quality_checks: list[dict[str, Any]] = []

for spec in SPECS:
    clean_df, metrics, checks = ETLPipelineRunner(spec).run()
    if clean_df is not None:
        datasets[spec.name] = clean_df
    run_metrics.append(metrics)
    quality_checks.extend(checks)

metrics_df = pd.DataFrame(run_metrics)
quality_df = pd.DataFrame(quality_checks)
metrics_df.to_csv(OUTPUT_DIR / "pipeline_metrics.csv", index=False)
quality_df.to_csv(OUTPUT_DIR / "data_quality_report.csv", index=False)

print(f"Completed {len(datasets)}/{len(SPECS)} datasets and wrote {sum(m['output_rows'] for m in run_metrics):,} rows.")
metrics_df


2026-06-20 14:14:45,580 [INFO] Extracted raw_orders.csv: 100000 rows, 11 columns
2026-06-20 14:14:46,566 [INFO] raw_orders: removed 0 duplicate key rows
2026-06-20 14:14:48,271 [INFO] Loaded 100000 rows to c:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\data\processed_v3\raw_orders.csv
2026-06-20 14:14:49,237 [INFO] Extracted raw_customers.json: 100000 rows, 6 columns
2026-06-20 14:14:50,050 [INFO] raw_customers: removed 5000 duplicate key rows
2026-06-20 14:14:50,851 [INFO] Loaded 95000 rows to c:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\data\processed_v3\raw_customers.csv
2026-06-20 14:14:51,239 [INFO] Extracted raw_products.csv: 100000 rows, 5 columns
2026-06-20 14:14:51,550 [WARNING] raw_products: quarantined 558 invalid rows
2026-06-20 14:14:51,644 [INFO] raw_products: removed 0 duplicate key rows
2026-06-20 14:14:52,505 [INFO] Loaded 99442 rows to c:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\data\processed_v3\raw_products.csv
2026-06-20 14:14:52,660 [INFO] Extracted orders.csv: 

Completed 10/10 datasets and wrote 839,940 rows.


,run_id,dataset,status,source_rows,output_rows,duplicates_removed,output,elapsed_seconds,error
0,99d145c8-a2b1-4ba1-b851-d69528dc5f17,raw_orders,SUCCESS,100000,100000,0,c:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\...,3.3605,None
1,a9662c01-24ef-4ce3-a854-5321bb5f4b33,raw_customers,SUCCESS,100000,95000,5000,c:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\...,2.5714,None
2,583671bb-2238-450b-ac9b-98c01c61809c,raw_products,SUCCESS,100000,99442,558,c:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\...,1.6363,None
3,c5bbac8c-ecad-4e7f-8bae-f1e64aa515ff,orders,SUCCESS,49909,49909,0,c:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\...,0.6052,None
4,973d3ba8-271e-4973-8a5b-cb21601f9d09,customers,SUCCESS,100000,96000,4000,c:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\...,1.9322,None
5,696ccfe9-4ed8-4837-b907-891ba3c1900e,products,SUCCESS,100000,99771,229,c:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\...,1.7351,None
6,9567f120-58cc-468f-a203-714e59fd660a,payments,SUCCESS,49909,49909,0,c:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\...,0.8582,None
7,fedb002e-ab28-441a-b0d2-101520b4c12d,locations,SUCCESS,100000,100000,0,c:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\...,1.2138,None
8,6829933d-2a79-4f4c-9fc8-aa4678554c37,shipping,SUCCESS,49909,49909,0,c:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\...,0.5357,None
9,471e3390-f7de-4245-a1a4-0f16b5406789,order_items,SUCCESS,100000,100000,0,c:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\...,0.7380,None


## Business enrichment and integrity checks

The normalized commerce model and the newer raw feed have different identifiers, so they produce separate facts. This avoids an invalid cross-system join while still making both feeds analysis-ready.


In [6]:
def add_integrity_check(name: str, child: pd.DataFrame, child_key: str, parent: pd.DataFrame, parent_key: str) -> None:
    missing = (~child[child_key].dropna().isin(parent[parent_key].dropna())).sum()
    quality_checks.append({"dataset": name, "check": f"foreign_key:{child_key}->{parent_key}",
                           "severity": "WARNING", "failed_rows": int(missing),
                           "status": "PASS" if missing == 0 else "FAIL"})


add_integrity_check("orders", datasets["orders"], "customer_id", datasets["customers"], "customer_id")
add_integrity_check("orders", datasets["orders"], "payment_id", datasets["payments"], "payment_id")
add_integrity_check("order_items", datasets["order_items"], "order_id", datasets["orders"], "order_id")
add_integrity_check("order_items", datasets["order_items"], "product_id", datasets["products"], "product_id")
add_integrity_check("shipping", datasets["shipping"], "order_id", datasets["orders"], "order_id")
add_integrity_check("shipping", datasets["shipping"], "location_id", datasets["locations"], "location_id")
add_integrity_check("raw_orders", datasets["raw_orders"], "customer_id", datasets["raw_customers"], "customer_id")
add_integrity_check("raw_orders", datasets["raw_orders"], "product_id", datasets["raw_products"], "product_id")

# Normalized commerce fact, one row per order item.
fact_orders = (datasets["order_items"]
    .merge(datasets["orders"], on="order_id", how="left", validate="many_to_one")
    .merge(datasets["products"], on="product_id", how="left", validate="many_to_one")
    .merge(datasets["payments"], on="payment_id", how="left", validate="many_to_one")
    .merge(datasets["customers"], on="customer_id", how="left", validate="many_to_one", suffixes=("", "_customer")))

shipping_detail = datasets["shipping"].merge(datasets["locations"], on="location_id", how="left", validate="many_to_one")
fact_orders = fact_orders.merge(shipping_detail, on="order_id", how="left", validate="many_to_one", suffixes=("", "_shipment"))
fact_orders["gross_revenue"] = (fact_orders["quantity"] * fact_orders["selling_price"]).round(2)
fact_orders["estimated_cost"] = (fact_orders["quantity"] * fact_orders["cost_price"]).round(2)
fact_orders["estimated_profit"] = (fact_orders["gross_revenue"] - fact_orders["estimated_cost"] - fact_orders["shipping_cost"].fillna(0)).round(2)
fact_orders["order_month"] = fact_orders["order_date"].dt.to_period("M").astype("string")

# New raw-feed fact, preserving line-item grain and applying percentage discounts.
fact_raw_orders = (datasets["raw_orders"]
    .merge(datasets["raw_products"][["product_id", "product_name", "category", "price"]], on="product_id", how="left", validate="many_to_one")
    .merge(datasets["raw_customers"], on="customer_id", how="left", validate="many_to_one"))
fact_raw_orders["gross_revenue"] = (fact_raw_orders["quantity"] * fact_raw_orders["price"]).round(2)
fact_raw_orders["discount_amount"] = (fact_raw_orders["gross_revenue"] * fact_raw_orders["discount"].fillna(0) / 100).round(2)
fact_raw_orders["net_revenue"] = (fact_raw_orders["gross_revenue"] - fact_raw_orders["discount_amount"]).round(2)
fact_raw_orders["order_month"] = fact_raw_orders["order_date"].dt.to_period("M").astype("string")

CSVLoader(OUTPUT_DIR / "fact_order_items.csv").load(fact_orders)
CSVLoader(OUTPUT_DIR / "fact_raw_orders.csv").load(fact_raw_orders)
print(f"Created fact_order_items ({len(fact_orders):,} rows) and fact_raw_orders ({len(fact_raw_orders):,} rows).")


2026-06-20 14:15:08,921 [INFO] Loaded 100000 rows to c:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\data\processed_v3\fact_order_items.csv
2026-06-20 14:15:11,775 [INFO] Loaded 100000 rows to c:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\data\processed_v3\fact_raw_orders.csv


Created fact_order_items (100,000 rows) and fact_raw_orders (100,000 rows).


In [7]:
monthly_commerce = (fact_orders.groupby(["order_month", "category"], dropna=False)
    .agg(orders=("order_id", "nunique"), units=("quantity", "sum"),
         gross_revenue=("gross_revenue", "sum"), estimated_profit=("estimated_profit", "sum"))
    .reset_index().sort_values(["order_month", "gross_revenue"], ascending=[True, False]))

monthly_raw = (fact_raw_orders.groupby(["order_month", "category"], dropna=False)
    .agg(orders=("order_id", "nunique"), units=("quantity", "sum"),
         gross_revenue=("gross_revenue", "sum"), discounts=("discount_amount", "sum"),
         net_revenue=("net_revenue", "sum"))
    .reset_index().sort_values(["order_month", "net_revenue"], ascending=[True, False]))

CSVLoader(OUTPUT_DIR / "monthly_commerce_summary.csv").load(monthly_commerce)
CSVLoader(OUTPUT_DIR / "monthly_raw_summary.csv").load(monthly_raw)

quality_df = pd.DataFrame(quality_checks)
quality_df.to_csv(OUTPUT_DIR / "data_quality_report.csv", index=False)

summary = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "datasets_processed": len(datasets),
    "source_rows_processed": int(sum(m["source_rows"] for m in run_metrics)),
    "clean_rows_written": int(sum(m["output_rows"] for m in run_metrics)),
    "quality_checks": len(quality_df),
    "quality_warnings": int(((quality_df["severity"] == "WARNING") & (quality_df["status"] == "FAIL")).sum()),
    "outputs": sorted(path.name for path in OUTPUT_DIR.glob("*.csv")),
}
with (OUTPUT_DIR / "run_summary.json").open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)

print(json.dumps(summary, indent=2))
monthly_raw.head(10)


2026-06-20 14:15:12,073 [INFO] Loaded 78 rows to c:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\data\processed_v3\monthly_commerce_summary.csv
2026-06-20 14:15:12,079 [INFO] Loaded 91 rows to c:\Users\Jolie Bwiza\Music\DE_pipeline_pro_02\data\processed_v3\monthly_raw_summary.csv


{
  "completed_at_utc": "2026-06-20T21:15:12.087510+00:00",
  "datasets_processed": 10,
  "source_rows_processed": 849727,
  "clean_rows_written": 839940,
  "quality_checks": 79,
  "quality_warnings": 10,
  "outputs": [
    "customers.csv",
    "data_quality_report.csv",
    "fact_order_items.csv",
    "fact_raw_orders.csv",
    "locations.csv",
    "monthly_commerce_summary.csv",
    "monthly_raw_summary.csv",
    "order_items.csv",
    "orders.csv",
    "payments.csv",
    "pipeline_metrics.csv",
    "products.csv",
    "raw_customers.csv",
    "raw_orders.csv",
    "raw_products.csv",
    "shipping.csv"
  ]
}


,order_month,category,orders,units,gross_revenue,discounts,net_revenue
4,2025-06,home & kitchen,518,3367,2211092.36,150535.51,2060556.85
5,2025-06,office supplies,528,3530,2109724.62,137193.66,1972530.96
1,2025-06,apparel,508,3239,2014147.09,133898.24,1880248.85
2,2025-06,books,504,3259,1951978.33,130819.69,1821158.64
3,2025-06,electronics,488,3063,1850746.72,125733.5,1725013.22
0,2025-06,accessories,503,3112,1762664.5,119577.19,1643087.31
6,2025-06,<NA>,93,513,0.0,0.0,0.0
9,2025-07,books,1197,7851,4662127.0,240090.35,4422036.65
8,2025-07,apparel,1173,7526,4639634.56,319619.8,4320014.76
11,2025-07,home & kitchen,1172,7425,4466009.88,273940.04,4192069.84


## Output contract

Successful execution writes cleaned source tables, enriched facts, monthly summaries, `pipeline_metrics.csv`, `data_quality_report.csv`, and `run_summary.json` beneath `data/processed_v3/`. Re-running the notebook is idempotent: outputs are replaced and never re-ingested.
